# Max Support Baseline

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
from contract import ROOT,REFERENCE,read_json,resolve
from experiment_supervision import ExperimentSupervisor
EXECUTE_NEW_RUN=False
ATTEMPT='attempt-01'
RESTART_OF=None
config_path=ROOT/'configs/experiments/max_support_baseline.yaml'
config=resolve(config_path,ATTEMPT,176)
print('Resolved decisions:',config['budget_decisions'],'evaluation episodes:',config['evaluation_episodes'])
if EXECUTE_NEW_RUN:
    result=ExperimentSupervisor().run(config_path,ATTEMPT,176,RESTART_OF)
    print('NEW RUN:',json.dumps(result,indent=2))
else:
    result=read_json(REFERENCE/'runs/P1-MAX/attempt-01/result.json')
    assert result['status']=='PASS'
    print('HISTORICAL RESULT; set EXECUTE_NEW_RUN=True for a fresh exclusive run:',result['status'])
print('Max Support Baseline definitions/execution completed.')


Frozen runtime contract definitions/execution completed.
Exclusive attempts and supervised worker processes definitions/execution completed.
Resolved decisions: 51200 evaluation episodes: 10
HISTORICAL RESULT; set EXECUTE_NEW_RUN=True for a fresh exclusive run: PASS
Max Support Baseline definitions/execution completed.
